# Day 6 — Pandas: Combining DataFrames

## Concept: `.merge()`

Real datasets rarely come in one clean file. You'll often have, say, a `students` DataFrame (name, score) and a separate `extra_info` DataFrame (name, year) — and you need to combine them into one DataFrame to work with both together.

```python
pd.merge(df1, df2, on="common_column", how="inner")
```

- `on` — the column both DataFrames share, used to match rows together (like a shared key)
- `how` — controls what happens when a match isn't found on one side:
  - `"inner"` — keep only rows where the key exists in **both** DataFrames (default)
  - `"left"` — keep all rows from the left DataFrame, fill with NaN if no match in the right
  - `"right"` — keep all rows from the right DataFrame, same idea reversed
  - `"outer"` — keep everything from both, NaN wherever there's no match

This is basically a SQL join — `on` is like matching a primary key/foreign key, and `how` maps directly to `INNER JOIN`, `LEFT JOIN`, `RIGHT JOIN`, `FULL OUTER JOIN`.

In [1]:
import pandas as pd
import numpy as np

## Exercise 1: Merge students with their extra info

Merge `students` and `extra_info` on `"name"`, keeping **all** students even if they don't have a matching year (missing year should be `NaN`).

In [2]:
students = pd.DataFrame({
    "name": ["Alice", "Bob", "Carol", "Dave"],
    "score": [85, 90, 78, 60]
})

extra_info = pd.DataFrame({
    "name": ["Alice", "Bob", "Carol"],
    "year": [1, 2, 1]
})

In [3]:
def merge_students(students, extra_info):
    """Merge students and extra_info on 'name', keeping all students
    even if they don't have a matching year (missing year should be NaN)."""
    return pd.merge(students, extra_info , on = "name", how = "left")
     

In [4]:
# Test cell
result = merge_students(students, extra_info)
print(result)

    name  score  year
0  Alice     85   1.0
1    Bob     90   2.0
2  Carol     78   1.0
3   Dave     60   NaN


## Exercise 2: `how="inner"` vs `how="left"`

Two departments have separate attendance records for the same week. Some students appear in only one department's list.

Write **two** functions:
- `only_students_in_both(dept_a, dept_b)` — return only students who appear in **both** lists
- `all_students_dept_a(dept_a, dept_b)` — return **all** students from `dept_a`, with `NaN` filled in for any missing info from `dept_b`

This is to make the `how="inner"` vs `how="left"` difference concrete — same two DataFrames, different `how`, different result.

In [5]:
dept_a = pd.DataFrame({
    "name": ["Alice", "Bob", "Eve"],
    "attendance": [0.9, 0.8, 0.95]
})

dept_b = pd.DataFrame({
    "name": ["Alice", "Bob", "Frank"],
    "club": ["Chess", "Debate", "Art"]
})

In [12]:
def only_students_in_both(dept_a, dept_b):
    """Return only students who appear in both dept_a and dept_b."""
    return pd.merge(dept_a, dept_b, on = "name", how = "inner")

def all_students_dept_a(dept_a, dept_b):
    """Return all students from dept_a, with NaN for missing dept_b info."""
    return pd.merge(dept_a, dept_b, on = "name", how = "left")

In [13]:
# Test cell
print(only_students_in_both(dept_a, dept_b))
print()
print(all_students_dept_a(dept_a, dept_b))

    name  attendance    club
0  Alice         0.9   Chess
1    Bob         0.8  Debate

    name  attendance    club
0  Alice        0.90   Chess
1    Bob        0.80  Debate
2    Eve        0.95     NaN


## Exercise 3: Mixed review — merge + filter with `notna()`

Merge `students` and `extra_info` (from Exercise 1) using `how="left"`, then return only the rows where `year` is **not missing** — using the `notna()` + boolean filtering pattern from Day 5 (not `dropna`).

In [ ]:
def merged_with_known_year(students, extra_info):
    """Merge students and extra_info (how='left'), then keep only rows
    where 'year' is not missing. Use notna() filtering, not dropna()."""

    Merged_df = pd.merge(students, extra_info, on = "name", how = "left")
    return Merged_df[Merged_df["year"].notna()]   
    # return Merged_df.dropna(subset = ["year"])


In [21]:
# Test cell
print(merged_with_known_year(students, extra_info))

    name  score  year
0  Alice     85   1.0
1    Bob     90   2.0
2  Carol     78   1.0


## Exercise 4: Mixed review — merge + groupby + agg

Merge `dept_a` and `dept_b` (from Exercise 2) using `how="outer"` (keep everyone from both sides), then group by `club` and find the **average attendance** per club.

Note: some rows won't have a `club` (NaN) — that's expected, `groupby` will just skip NaN groups by default.

In [ ]:
def avg_attendance_by_club(dept_a, dept_b):
    """Merge dept_a and dept_b (how='outer'), then group by 'club'
    and return the average attendance per club."""
    m_df = pd.merge(dept_a, dept_b, on = "name", how =  "outer")
    return m_df.groupby("club")["attendance"].mean()

In [23]:
# Test cell
print(avg_attendance_by_club(dept_a, dept_b))

club
Art       NaN
Chess     0.9
Debate    0.8
Name: attendance, dtype: float64
